# Preparação dos dados do INMET

Este notebook lê os arquivos horários das estações meteorológicas do INMET e produz uma tabela com uma linha por dia para o Sul do país. Primeiro, cada estação é agregada de horário para diário, e depois as estações são resumidas em um valor regional por dia, já que o ONS é analisado no nível do subsistema e não de uma cidade. Os arquivos originais ficam em data/raw/INMET, com uma pasta por ano e um CSV por estação.

In [41]:
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW = Path("../data/raw/INMET")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

ANOS = range(2021, 2027)
UFS = {"PR", "SC", "RS"}
INICIO = "2021-01-01"
FIM = "2025-12-31"

UTC_PARA_BRASILIA = pd.Timedelta(hours=-3)
MIN_HORAS_VALIDAS = 18
COBERTURA_MINIMA = 0.3

### Escolha das estações

O nome de cada arquivo já traz a região e o estado, como em INMET_S_PR_A807_CURITIBA. Os arquivos que começam com INMET_S são as estações do Subsistema Sul, ou seja, do Paraná, Santa Catarina e Rio Grande do Sul, que formam o nosso recorte.

In [42]:
# Lista os arquivos de estações da região Sul nos anos pedidos.
def listar_arquivos_sul(anos):
    arquivos = []
    
    for ano in anos:
        arquivos += sorted((RAW / str(ano)).glob("INMET_S_*.CSV"))

    return arquivos


arquivos = listar_arquivos_sul(ANOS)
print(len(arquivos), "arquivos de estação")

641 arquivos de estação


### Leitura dos datasets

As oito primeiras linhas dos datasets do INMET são metadados da estação, portanto os dados começam na nona linha. Utilizaremos as colunas chuva, temperatura e rajada de vento, que são as ligadas às tempestades. As demais colunas, como pressão, umidade e radiação, foram desconsideradas.

As horas do arquivo estão em UTC, e o dia local de Brasília começa às 03:00 UTC. Se agregássemos pela data UTC, a chuva da noite cairia no dia seguinte, e o ONS registra seus eventos no horário local. Por isso, os horários foram convertidos para UTC-3 antes da extração da data. O Brasil não tem horário de verão desde 2019, então um deslocamento fixo de três horas é suficiente para o período selecionado.

Valores espúrios são marcados como valores ausentes: chuva acima de 200 mm em uma hora; temperatura de -10 a 45 graus; e rajadas de vento acima de 60 m/s são removidos e marcados como ausentes.

In [43]:
VARIAVEIS = {
    "chuva": "PRECIPITACAO TOTAL",
    "temperatura": "TEMPERATURA DO AR",
    "rajada": "VENTO, RAJADA",
}

LIMITES = {"chuva": (0, 200), "temperatura": (-10, 45), "rajada": (0, 60)}


# Remove acentos e padroniza caixa para comparar nomes de colunas.
def normalizar(texto):
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sem_acento.upper().strip()


# Lê um arquivo horário e devolve a tabela limpa em UTC e local, mais as contagens de problemas.
def ler_arquivo(caminho):
    tabela = pd.read_csv(caminho, sep=";", skiprows=8, decimal=",", encoding="latin-1")
    colunas = {normalizar(nome): nome for nome in tabela.columns}
    contagens = {"linhas": len(tabela)}

    data = pd.to_datetime(tabela.iloc[:, 0].astype(str).str.replace("/", "-"), format="%Y-%m-%d", errors="coerce")
    hora = pd.to_numeric(tabela.iloc[:, 1].astype(str).str[:2], errors="coerce")
    horario_utc = data + pd.to_timedelta(hora, unit="h")

    horario = pd.DataFrame({"horario_utc": horario_utc})
    horario["data"] = (horario_utc + UTC_PARA_BRASILIA).dt.normalize()

    for variavel, prefixo in VARIAVEIS.items():
        nome = next((original for normal, original in colunas.items() if normal.startswith(prefixo)), None)
        if nome is None:
            raise ValueError(f"{caminho.name}: coluna de {variavel} não encontrada")

        valores = pd.to_numeric(tabela[nome], errors="coerce")
        minimo, maximo = LIMITES[variavel]
        fora_da_faixa = (valores < minimo) | (valores > maximo)

        contagens[f"{variavel}_ausente"] = int(valores.isna().sum())
        contagens[f"{variavel}_fora_da_faixa"] = int(fora_da_faixa.sum())
        horario[variavel] = valores.where(~fora_da_faixa)

    contagens["datas_invalidas"] = int(horario["horario_utc"].isna().sum())
    horario = horario.dropna(subset=["horario_utc"])
    contagens["horarios_repetidos"] = int(horario.duplicated("horario_utc").sum())
    
    return horario.drop_duplicates("horario_utc"), contagens

### Agregação por dia em cada estação

Cada arquivo é agregado para diário logo depois de lido. A chuva do dia é a soma das precipitações por hora, a temperatura máxima e a mínima vêm dos extremos horários, e a rajada do dia é o máximo registrado, já que vento forte é um pico e não uma média. A temperatura horária pode subestimar um pouco os extremos reais, mas a temperatura não é a variável central do problema, e isso evita lidar com as colunas de máxima e mínima da hora anterior.

Um dia só vale se tiver pelo menos 18 horas válidas da variável. A escolha do total de horas para o corte é arbitrária e pode sofrer mudanças. 

In [44]:
# Resume as leituras horárias de uma estação em um registro por dia.
def agregar_por_dia(horario):
    grupos = horario.groupby("data")
    diario = pd.DataFrame({
        "chuva": grupos["chuva"].sum(min_count=1),
        "temp_max": grupos["temperatura"].max(),
        "temp_min": grupos["temperatura"].min(),
        "rajada_max": grupos["rajada"].max(),
    })

    horas_validas = grupos[list(VARIAVEIS)].count()
    origem = {"chuva": "chuva", "temp_max": "temperatura", "temp_min": "temperatura", "rajada_max": "rajada"}
    
    for coluna, variavel in origem.items():
        diario[coluna] = diario[coluna].where(horas_validas[variavel] >= MIN_HORAS_VALIDAS)
    
    return diario.reset_index()

diarios, contagens = [], []

for caminho in arquivos:
    horario, contagem = ler_arquivo(caminho)
    _, _, uf, codigo, *_ = caminho.stem.split("_")
    diario = agregar_por_dia(horario)
    diario["uf"], diario["estacao"] = uf, codigo
    diarios.append(diario)
    contagens.append(contagem)

diario_estacoes = pd.concat(diarios, ignore_index=True)
diario_estacoes = diario_estacoes[diario_estacoes["data"].between(INICIO, FIM)]
print(len(diario_estacoes), "registros estação-dia")

175194 registros estação-dia


### Qualidade dos dados

Atabela abaixo mostra quanto de cada variável estava ausente nos arquivos originais e quanto passa a estar ausente depois de tratarmos os valores fora da faixa estabelecida.

In [45]:
total = pd.DataFrame(contagens).sum()

qualidade = pd.DataFrame({
    "ausente original (%)": [100 * total[f"{v}_ausente"] / total["linhas"] for v in VARIAVEIS],
    "ausente após tratamento (%)": [
        100 * (total[f"{v}_ausente"] + total[f"{v}_fora_da_faixa"]) / total["linhas"] for v in VARIAVEIS
    ],
}, index=list(VARIAVEIS)).round(2)

display(qualidade)
print("datas inválidas:", total["datas_invalidas"], "| horários repetidos removidos:", total["horarios_repetidos"])

,ausente original (%),ausente após tratamento (%)
chuva,25.94,25.94
temperatura,22.30,22.30
rajada,27.99,27.99


datas inválidas: 0 | horários repetidos removidos: 0


### Seleção das estações

A rede de estações do INMET cresce com os anos, então as estações mais novas só têm dados em parte do período. Por isso, mantemos apenas as estações com pelo menos 30% dos dias válidos. A tabela abaixo ilustra como a cobertura se distribui entre estados.

In [46]:
calendario = pd.date_range(INICIO, FIM, freq="D", name="data")

dia_valido = diario_estacoes["chuva"].notna() & diario_estacoes["temp_max"].notna()
cobertura = dia_valido.groupby([diario_estacoes["uf"], diario_estacoes["estacao"]]).sum() / len(calendario)

estacoes = cobertura.rename("cobertura").reset_index()
estacoes["selecionada"] = estacoes["cobertura"] >= COBERTURA_MINIMA

display(estacoes.groupby("uf", as_index=False)["selecionada"].agg(total="size", selecionadas="sum"))

,uf,total,selecionadas
0,PR,26,22
1,RS,82,45
2,SC,24,22


### Agrupamento de estações

As estações foram agrupadas por dia e agregadas com os valores de chuva, temperatura e rajada de vento. A média de chuva é utilizada para a região inteira, mas pode diminuir o impactor de uma tempestade que atinge só uma parte dela, então registramos também o máximo entre estações. Para a temperatura, usamos médias, e para o vento apenas o máximo da rajada. 

O Rio Grande do Sul tem mais estações que os outros dois estados, então a média simples pesa mais esse estado. Foi realizada a média das médias por estado para verificar a diferença média absoluta (MAE) e correlação dos dados agrupados por estado.

In [47]:
selecionadas = diario_estacoes.merge(estacoes.loc[estacoes["selecionada"], ["uf", "estacao"]])

clima = selecionadas.groupby("data").agg(
    n_estacoes=("estacao", "nunique"),
    chuva_media=("chuva", "mean"),
    chuva_max=("chuva", "max"),
    temp_max_media=("temp_max", "mean"),
    temp_min_media=("temp_min", "mean"),
    rajada_max=("rajada_max", "max"),
).reindex(calendario)

print("dias sem nenhuma estação:", (clima["n_estacoes"] == 0).sum())

selecionadas_por_uf = selecionadas.groupby(["data", "uf"])[["chuva", "temp_max", "temp_min"]].mean().groupby("data").mean()
sensibilidade = pd.DataFrame({
    "correlação": [
        clima["chuva_media"].corr(selecionadas_por_uf["chuva"]),
        clima["temp_max_media"].corr(selecionadas_por_uf["temp_max"]),
        clima["temp_min_media"].corr(selecionadas_por_uf["temp_min"]),
    ],
    "MAE": [
        (clima["chuva_media"] - selecionadas_por_uf["chuva"]).abs().mean(),
        (clima["temp_max_media"] - selecionadas_por_uf["temp_max"]).abs().mean(),
        (clima["temp_min_media"] - selecionadas_por_uf["temp_min"]).abs().mean(),
    ],
}, index=["chuva (mm)", "temp. máxima (°C)", "temp. mínima (°C)"]).round(3)

clima["n_estacoes"] = clima["n_estacoes"].fillna(0).astype(int)
display(sensibilidade)

dias sem nenhuma estação: 0


,correlação,MAE
chuva (mm),0.957,1.058
temp. máxima (°C),0.993,0.494
temp. mínima (°C),0.995,0.354


É possível observar que a correlação entre a quantidade de chuva de cada estado é ligeiramente menor em relação à temperatura, e a MAE da quantidade de chuva é maior. Entretanto, devido à alta correlação e a baixa MAE, será mantida a média simples.

### Exportação

A base final é exportada em Parquet. Após a preparação dos dados, nenhum valor ausente se manteve.

In [48]:
colunas_clima = clima.columns.drop("n_estacoes")
clima[colunas_clima] = clima[colunas_clima].ffill(limit=2)

print("valores ausentes:", clima.isna().sum()[lambda x : x > 0].count())

clima.reset_index().to_parquet(INTERIM / "inmet_diario_sul.parquet", index=False)

valores ausentes: 0
